In [1]:
import pandas as pd
import numpy as np
import umap

df = pd.read_csv('../data/processed/ukbms_sites_with_embeddings.csv')
emb_cols = [c for c in df.columns if c.startswith('emb_')]
X = df[emb_cols].values
print(X.shape)

(2921, 128)


In [ ]:
reducer = umap.UMAP(n_neighbors=15, min_dist=0.3, random_state=42)
coords = reducer.fit_transform(X)
df['umap_x'], df['umap_y'] = coords[:, 0], coords[:, 1]
df.to_csv('../data/processed/umap_result.csv', index=False)

In [3]:
groups = {}
for st in ['UKBMS', 'WCBS']:
    sub = df[df['Survey_type'] == st]
    groups[st] = {
        'centroid': (sub['umap_x'].mean(), sub['umap_y'].mean()),
        'spread_x': sub['umap_x'].std(),
        'spread_y': sub['umap_y'].std(),
        'n': len(sub),
    }
    print(f"{st}: n={groups[st]['n']}, centroid={groups[st]['centroid']}, "
          f"spread=({groups[st]['spread_x']:.2f}, {groups[st]['spread_y']:.2f})")

# Distance between the two group centers
cx1, cy1 = groups['UKBMS']['centroid']
cx2, cy2 = groups['WCBS']['centroid']
centroid_dist = np.sqrt((cx1-cx2)**2 + (cy1-cy2)**2)

# Average internal spread, as a yardstick to compare against
avg_spread = np.mean([groups['UKBMS']['spread_x'], groups['UKBMS']['spread_y'],
                       groups['WCBS']['spread_x'], groups['WCBS']['spread_y']])

print(f"\nCentroid distance: {centroid_dist:.2f}")
print(f"Average within-group spread: {avg_spread:.2f}")
print(f"Ratio: {centroid_dist / avg_spread:.2f}")

UKBMS: n=2015, centroid=(np.float32(1.9003489), np.float32(1.0457337)), spread=(2.65, 2.55)
WCBS: n=906, centroid=(np.float32(4.3064413), np.float32(2.312446)), spread=(3.61, 2.28)

Centroid distance: 2.72
Average within-group spread: 2.77
Ratio: 0.98


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import numpy as np

emb_cols = [c for c in df.columns if c.startswith('emb_')]
X = df[emb_cols].values
y = (df['Survey_type'] == 'UKBMS').astype(int)  # 1 = UKBMS, 0 = WCBS

# Baseline: if you just always guessed the majority class, what accuracy
# would that get you "for free"? Any real classifier needs to beat this.
majority_baseline = max(y.mean(), 1 - y.mean())
print(f"Majority-class baseline accuracy: {majority_baseline:.3f}")

clf = LogisticRegression(max_iter=1000)
scores = cross_val_score(clf, X, y, cv=5, scoring='accuracy')
print(f"5-fold CV accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")

Majority-class baseline accuracy: 0.690
5-fold CV accuracy: 0.761 (+/- 0.015)


In [5]:
from sklearn.model_selection import permutation_test_score

score, perm_scores, pvalue = permutation_test_score(
    clf, X, y, cv=5, n_permutations=100, scoring='accuracy', n_jobs=-1
)
print(f"Real score: {score:.3f}")
print(f"Mean permuted score: {perm_scores.mean():.3f}")
print(f"p-value: {pvalue:.4f}")

Real score: 0.761
Mean permuted score: 0.665
p-value: 0.0099


In [6]:
print(df.groupby('Survey_type')[['lat','lon']].agg(['mean','std']))

# Sharper test: can the classifier ALSO separate them using just lat/lon
# (no embedding at all)? If yes, geography alone explains a chunk of your result.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

X_geo = df[['lat','lon']].values
clf_geo = LogisticRegression(max_iter=1000)
scores_geo = cross_val_score(clf_geo, X_geo, y, cv=5, scoring='accuracy')
print(f"Lat/lon-only accuracy: {scores_geo.mean():.3f}")

                   lat                 lon          
                  mean       std      mean       std
Survey_type                                         
UKBMS        52.370047  1.594944 -1.646446  1.595642
WCBS         52.326635  1.504387 -1.502444  1.600879
Lat/lon-only accuracy: 0.690


In [ ]:
from sklearn.model_selection import cross_val_predict

# Out-of-fold predictions, not clf_final: clf_final saw every site during
# training, so its own predictions would look artificially accurate. These
# come from the same 5-fold split as the reported 76.1% CV accuracy above,
# so each site's probability comes from a fold that never trained on it.
proba = cross_val_predict(clf, X, y, cv=5, method='predict_proba')
pred = (proba[:, 1] >= 0.5).astype(int)

df['p_ukbms'] = proba[:, 1]
df['pred_correct'] = (pred == y).astype(int)

oof_acc = df['pred_correct'].mean()
print(f"Out-of-fold accuracy: {oof_acc:.3f} (should match the 5-fold CV accuracy above)")

df.to_csv('../data/processed/umap_result.csv', index=False)

In [7]:
import joblib

# Retrain on the full dataset (not just cross-val folds) for reuse
clf_final = LogisticRegression(max_iter=1000)
clf_final.fit(X, y)

joblib.dump(clf_final, '../src/habitat_classifier.joblib')
joblib.dump(reducer, '../src/umap_reducer.joblib')
print("Saved classifier and reducer for reuse")

Saved classifier and reducer for reuse
